# OpenClaw Colab LLM Launcher

Run these cells from top to bottom. The notebook pulls the latest launcher scripts from GitHub, builds native `llama-server`, starts the model with `--jinja`, exposes it through Cloudflare, and checks whether the endpoint returns real OpenAI-style `tool_calls`.

In [5]:
# 1. Clone or update the launcher repo
REPO_URL = "https://github.com/Cobster-10/custom_llm.git"
REPO_DIR = "/content/custom_llm"

import os
from pathlib import Path

if Path(REPO_DIR, ".git").exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 15 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 7.97 KiB | 2.66 MiB/s, done.
From https://github.com/Cobster-10/custom_llm
   1143685..c0a2b07  main       -> origin/main
Updating 1143685..c0a2b07
Fast-forward
 .gitattributes                     |   5 +
 .gitignore                         |  24 ++++
 README.md                          | Bin 30 -> 1419 bytes
 colab_launcher.ipynb               | 182 +++++++++++++++++++++++++
 config/model.env.example           |  16 +++
 llm_notebook (3).ipynb             | 264 +++++++++++++++++--------------------
 scripts/setup_colab.sh             |  50 +++++++
 scripts/start_cloudflare_tunnel.sh |  45 +++++++
 scripts/start_llama_server.sh      |  66 ++++++++++
 tests/test_tool_call.py            | 140 ++++++++++++++++++++
 10 files changed, 648 insertions(+), 14

In [6]:
# 2. Choose model/runtime settings
# Q8_K_P is the 8-bit quantization and fits large-memory Colab GPUs.
# If Colab gives you less VRAM, try Q4_K_M, IQ3_M, or IQ2_M.
%env MODEL_REPO=HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
%env MODEL_QUANT=Q8_K_P
%env PORT=8080
%env CTX_SIZE=131072
%env N_GPU_LAYERS=99
%env PARALLEL=1

env: MODEL_REPO=HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
env: MODEL_QUANT=Q8_K_P
env: PORT=8080
env: CTX_SIZE=131072
env: N_GPU_LAYERS=99
env: PARALLEL=1


In [7]:
# 3. Install dependencies, build llama.cpp with CUDA, and install cloudflared
!bash scripts/setup_colab.sh

==> Installing system packages
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]       
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,765 kB]
Get:12 http://archive.ubuntu.com/ubuntu jamm

In [8]:
# 4. Start llama-server in the background
!bash scripts/start_llama_server.sh

==> Starting llama-server
Model: HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:Q8_K_P
Local API: http://127.0.0.1:8080/v1
Log file: /content/llama-server.log
Command: /content/llama.cpp/build/bin/llama-server -hf HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:Q8_K_P --host 0.0.0.0 --port 8080 --jinja -c 131072 -ngl 99 --parallel 1
PID: 20224
Waiting briefly for the server to bind...
0.00.438.296 I log_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.438.303 I device_info:
0.00.686.367 I   - CUDA0   : NVIDIA A100-SXM4-80GB (81152 MiB, 80724 MiB free)
0.00.686.389 I   - CPU     : Intel(R) Xeon(R) CPU @ 2.20GHz (171061 MiB, 171061 MiB free)
0.00.686.538 I system_info: n_threads = 6 (n_threads_batch = 6) / 12 | CUDA : ARCHS = 800 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | AVX512_VNNI = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
0.00.686.605 I srv          init

In [ ]:
# 5. Wait until llama-server reports that the model is loaded
!python tests/wait_for_server.py --base-url http://127.0.0.1:8080 --timeout 1800 --log-file /content/llama-server.log

In [10]:
# 6. Local smoke test: does the server return structured OpenAI tool_calls?
!python tests/test_tool_call.py --base-url http://127.0.0.1:8080/v1 --model "$MODEL_REPO:$MODEL_QUANT"

POST http://127.0.0.1:8080/v1/chat/completions
HTTP 404



In [ ]:
# 7. Start Cloudflare tunnel and print the public OpenAI-compatible base URL
!bash scripts/start_cloudflare_tunnel.sh

In [ ]:
# 8. Public smoke test. Use this same base URL in OpenClaw if it passes.
from pathlib import Path

base_url_path = Path('/content/openclaw_base_url.txt')
if not base_url_path.exists():
    raise RuntimeError('No Cloudflare URL found. Re-run the tunnel cell first.')

public_base_url = base_url_path.read_text().strip()
print('OpenClaw base URL:', public_base_url)
!python tests/test_tool_call.py --base-url {public_base_url} --model "$MODEL_REPO:$MODEL_QUANT"

In [ ]:
# Optional: stop background services
!test -f /content/cloudflared.pid && kill $(cat /content/cloudflared.pid) || true
!test -f /content/llama-server.pid && kill $(cat /content/llama-server.pid) || true